# K-Nearest Neighbors (KNN) Implementation using PyTorch

This notebook demonstrates KNN algorithm implementation using PyTorch framework.

## 1. Import Required Libraries

In [1]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch Version: {torch.__version__}')

PyTorch Version: 2.10.0+cpu


## 2. Load and Explore Dataset
We'll use the Iris dataset - a classic multiclass classification dataset.

In [ ]:
# Load Iris dataset
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f'Dataset Shape: {X.shape}')
print(f'Number of Classes: {len(np.unique(y))}')
print(f'Feature Names: {feature_names}')
print(f'Target Names: {target_names}')

# Create DataFrame for visualization
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y
df['species'] = df['target'].map({i: name for i, name in enumerate(target_names)})
df.head()

In [ ]:
# Visualize data distribution
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for idx, feature in enumerate(feature_names):
    ax = axes[idx//2, idx%2]
    for target, name in enumerate(target_names):
        ax.hist(df[df['target']==target][feature], alpha=0.6, label=name, bins=15)
    ax.set_xlabel(feature)
    ax.set_ylabel('Frequency')
    ax.legend()
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_scaled)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_train_tensor = torch.LongTensor(y_train)
y_test_tensor = torch.LongTensor(y_test)

print(f'Training set size: {X_train_tensor.shape}')
print(f'Test set size: {X_test_tensor.shape}')

## 4. KNN Implementation using PyTorch

In [ ]:
class KNN_PyTorch:
    def __init__(self, k=3, distance_metric='euclidean'):
        self.k = k
        self.distance_metric = distance_metric
        self.X_train = None
        self.y_train = None

    def fit(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train

    def compute_distance(self, x1, x2):
        if self.distance_metric == 'euclidean':
            return torch.sqrt(torch.sum((x1 - x2) ** 2, dim=1))
        elif self.distance_metric == 'manhattan':
            return torch.sum(torch.abs(x1 - x2), dim=1)
        elif self.distance_metric == 'cosine':
            dot_product = torch.sum(x1 * x2, dim=1)
            norm_x1 = torch.sqrt(torch.sum(x1 ** 2, dim=1))
            norm_x2 = torch.sqrt(torch.sum(x2 ** 2, dim=1))
            return 1 - (dot_product / (norm_x1 * norm_x2 + 1e-8))

    def predict(self, X_test):
        predictions = []

        for test_point in X_test:
            # Compute distances to all training points
            distances = self.compute_distance(self.X_train, test_point.unsqueeze(0))

            # Get k nearest neighbors
            k_indices = torch.topk(distances, self.k, largest=False).indices
            k_nearest_labels = self.y_train[k_indices]

            # Majority voting
            unique_labels, counts = torch.unique(k_nearest_labels, return_counts=True)
            predicted_label = unique_labels[torch.argmax(counts)]
            predictions.append(predicted_label.item())

        return torch.tensor(predictions)

print('KNN class implemented successfully!')

## 5. Train and Evaluate KNN

In [ ]:
# Train KNN with k=5
knn = KNN_PyTorch(k=5, distance_metric='euclidean')
knn.fit(X_train_tensor, y_train_tensor)

# Make predictions
y_pred = knn.predict(X_test_tensor)

# Calculate accuracy
accuracy = torch.sum(y_pred == y_test_tensor).item() / len(y_test_tensor)
print(f'Accuracy with k=5: {accuracy:.4f}')

# Confusion Matrix
cm = confusion_matrix(y_test_tensor.numpy(), y_pred.numpy())
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix (k=5)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Classification Report
print('\nClassification Report:')
print(classification_report(y_test_tensor.numpy(), y_pred.numpy(), target_names=target_names))

## 6. Hyperparameter Tuning - Finding Optimal K

In [ ]:
# Test different k values
k_values = range(1, 21)
accuracies = []

for k in k_values:
    knn = KNN_PyTorch(k=k)
    knn.fit(X_train_tensor, y_train_tensor)
    y_pred = knn.predict(X_test_tensor)
    accuracy = torch.sum(y_pred == y_test_tensor).item() / len(y_test_tensor)
    accuracies.append(accuracy)

# Plot accuracy vs k
plt.figure(figsize=(10, 6))
plt.plot(k_values, accuracies, marker='o', linewidth=2, markersize=8)
plt.xlabel('K Value', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('KNN Accuracy vs K Value', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(k_values)
plt.show()

optimal_k = k_values[np.argmax(accuracies)]
print(f'Optimal k value: {optimal_k} with accuracy: {max(accuracies):.4f}')

## 7. Compare Distance Metrics

In [ ]:
distance_metrics = ['euclidean', 'manhattan', 'cosine']
results = {}

for metric in distance_metrics:
    knn = KNN_PyTorch(k=5, distance_metric=metric)
    knn.fit(X_train_tensor, y_train_tensor)
    y_pred = knn.predict(X_test_tensor)
    accuracy = torch.sum(y_pred == y_test_tensor).item() / len(y_test_tensor)
    results[metric] = accuracy
    print(f'{metric.capitalize()} Distance - Accuracy: {accuracy:.4f}')

# Visualize comparison
plt.figure(figsize=(10, 6))
plt.bar(results.keys(), results.values(), color=['skyblue', 'lightcoral', 'lightgreen'])
plt.xlabel('Distance Metric', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('KNN Performance with Different Distance Metrics (k=5)', fontsize=14)
plt.ylim([0, 1])
for i, (metric, acc) in enumerate(results.items()):
    plt.text(i, acc + 0.02, f'{acc:.4f}', ha='center', fontsize=11)
plt.show()

## 8. Additional Dataset - Wine Classification

In [ ]:
# Load Wine dataset
wine = load_wine()
X_wine, y_wine = wine.data, wine.target

# Split and preprocess
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(X_wine, y_wine, test_size=0.2, random_state=42)
scaler_w = StandardScaler()
X_train_w_scaled = scaler_w.fit_transform(X_train_w)
X_test_w_scaled = scaler_w.transform(X_test_w)

# Convert to tensors
X_train_w_tensor = torch.FloatTensor(X_train_w_scaled)
X_test_w_tensor = torch.FloatTensor(X_test_w_scaled)
y_train_w_tensor = torch.LongTensor(y_train_w)
y_test_w_tensor = torch.LongTensor(y_test_w)

# Train and evaluate
knn_wine = KNN_PyTorch(k=7)
knn_wine.fit(X_train_w_tensor, y_train_w_tensor)
y_pred_wine = knn_wine.predict(X_test_w_tensor)
accuracy_wine = torch.sum(y_pred_wine == y_test_w_tensor).item() / len(y_test_w_tensor)

print(f'Wine Dataset Classification Accuracy: {accuracy_wine:.4f}')
print(f'\nDataset Info:')
print(f'Features: {X_wine.shape[1]}')
print(f'Classes: {len(np.unique(y_wine))}')
print(f'Samples: {X_wine.shape[0]}')

## 9. Key Takeaways

1. **KNN Algorithm**: A non-parametric, lazy learning algorithm that classifies based on nearest neighbors
2. **Distance Metrics**: Choice of distance metric significantly impacts performance
3. **Hyperparameter K**: Optimal k value depends on the dataset and should be tuned
4. **PyTorch Implementation**: Efficient computation using tensor operations
5. **Scalability**: KNN can be slow with large datasets as it requires distance computation for all points

**Advantages:**
- Simple and intuitive
- No training phase
- Effective for small to medium datasets

**Disadvantages:**
- Computationally expensive for large datasets
- Sensitive to feature scaling
- Curse of dimensionality